# 02 · Data Quality & Preprocessing Study
**Telco Customer Churn Prediction with SQL and Machine Learning**  
*Section 6* · MSc Data Science project · Dr. Spoorthi H S

**In this notebook**
- Missing-value audit and domain-specific imputation of TotalCharges
- Outlier detection with IQR, Z-score and Isolation Forest
- Scaling, power transforms, skewness and kurtosis comparisons
- Multicollinearity (VIF), correlation pruning and PCA

> **How to use these notebooks.** The outputs below come from one complete run in Google Colab. Each part builds on objects created earlier (the SQL database, the prepared DataFrame, trained models), so the part notebooks are for reading and review. To reproduce the results, run [`00_full_pipeline_run_all.ipynb`](00_full_pipeline_run_all.ipynb) top to bottom.

[← Data Loading & SQL Pipeline](01_data_and_sql_pipeline.ipynb) · [Feature Engineering & Exploratory Analysis →](03_feature_engineering_and_eda.ipynb)

---

## 6. ADVANCED DATA PREPROCESSING

In [43]:
print('Data Quality Assessment:')
print(f'Missing Values:\n{df.isnull().sum()}')
print(f'\nData Types:\n{df.dtypes}')

# Handle missing values
df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

# Outlier detection - IQR method
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    if col != 'SeniorCitizen':
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print('✓ Preprocessing complete')


Data Quality Assessment:
Missing Values:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
Churn                0
MonthlyCharges       0
TotalCharges        11
Contract             0
PaperlessBilling     0
PaymentMethod        0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
dtype: int64

Data Types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
Churn                object
MonthlyCharges      float64
TotalCharges        float64
Contract             object
PaperlessBilling     object
PaymentMethod        object
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
On

**SECTION 6.1: PREPROCESSING TECHNIQUES STUDY (6 TECHNIQUES)**

The cells below **explore and compare** six families of preprocessing techniques on the numeric features (`tenure`, `MonthlyCharges`, `TotalCharges`) and report their effect:

1. Imputation strategies - statistical (mean/median), domain-specific (`TotalCharges = MonthlyCharges × tenure`), categorical mode
2. Outlier detection & treatment - IQR capping, Z-score, Isolation Forest
3. Feature scaling - Standard, Robust, MinMax, log transform
4. Distribution normalisation - Yeo-Johnson, Box-Cox, Quantile transformer
5. Multicollinearity & dimensionality - VIF, correlation pruning, PCA
6. Skewness & kurtosis correction - square root, log, reciprocal

> **How this feeds the model.** These comparisons run on copies of the data to inform decisions. The modelling pipeline itself uses the domain-specific imputation and IQR check from Section 6, one-hot encoding and engineered features (Section 7), and a `StandardScaler` fitted on the **training split only** (Section 9).

In [44]:
# ============================================================
# TECHNIQUE 1: MULTIPLE IMPUTATION STRATEGIES
# ============================================================
print('\n' + '='*60)
print('TECHNIQUE 1: MULTIPLE IMPUTATION STRATEGIES')
print('='*60)

df_imputed = df.copy()

# Strategy 1A: Statistical Imputation (Mean/Median)
print('\n1A. Statistical Imputation:')
numeric_cols = df_imputed.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    if df_imputed[col].isnull().any():
        # Use median for skewed distributions, mean for normal
        if df_imputed[col].skew() > 1 or df_imputed[col].skew() < -1:
            df_imputed[col].fillna(df_imputed[col].median(), inplace=True)
            print(f'  {col}: Median imputation (skewed distribution)')
        else:
            df_imputed[col].fillna(df_imputed[col].mean(), inplace=True)
            print(f'  {col}: Mean imputation (normal distribution)')

# Strategy 1B: Domain-specific Imputation
print('\n1B. Domain-Specific Imputation:')
if 'TotalCharges' in df_imputed.columns:
    # For TotalCharges, use tenure-based calculation
    mask = df_imputed['TotalCharges'].isnull()
    if mask.any():
        df_imputed.loc[mask, 'TotalCharges'] = df_imputed.loc[mask, 'MonthlyCharges'] * df_imputed.loc[mask, 'tenure']
        print(f'  TotalCharges: Calculated as MonthlyCharges × Tenure ({mask.sum()} records)')

# Strategy 1C: Forward Fill for Categorical
print('\n1C. Categorical Imputation (Mode + Forward Fill):')
cat_cols = df_imputed.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df_imputed[col].isnull().any():
        df_imputed[col].fillna(df_imputed[col].mode()[0], inplace=True)
        print(f'  {col}: Filled with mode value')

print('\n✓ Imputation complete. Missing values:', df_imputed.isnull().sum().sum())



TECHNIQUE 1: MULTIPLE IMPUTATION STRATEGIES

1A. Statistical Imputation:

1B. Domain-Specific Imputation:

1C. Categorical Imputation (Mode + Forward Fill):

✓ Imputation complete. Missing values: 0


In [45]:
# ============================================================
# TECHNIQUE 2: ROBUST OUTLIER DETECTION & TREATMENT (3 METHODS)
# ============================================================
print('\n' + '='*60)
print('TECHNIQUE 2: ROBUST OUTLIER DETECTION (3 METHODS)')
print('='*60)

from sklearn.ensemble import IsolationForest
from scipy.stats import zscore

# --- FIX START ---
# The previous `df` from `load_sql_data` was empty, which made `df_imputed` empty.
# To ensure this cell runs correctly, we'll re-initialize `df_outlier` from `raw_data`
# (which is correctly loaded from the CSV) and apply the necessary type conversions
# and imputations that `preprocessing` and `TECHNIQUE 1` cells were meant to handle.

df_outlier = raw_data.copy()

# Ensure numeric columns have correct data types and handle potential NaNs.
# These steps are often performed in initial data loading/preprocessing.
df_outlier['TotalCharges'] = pd.to_numeric(df_outlier['TotalCharges'], errors='coerce')
df_outlier['MonthlyCharges'] = pd.to_numeric(df_outlier['MonthlyCharges'], errors='coerce')
df_outlier['tenure'] = pd.to_numeric(df_outlier['tenure'], errors='coerce')
df_outlier['SeniorCitizen'] = pd.to_numeric(df_outlier['SeniorCitizen'], errors='coerce').fillna(0).astype(int)

# Apply imputation strategies for numeric columns, mimicking 'TECHNIQUE 1'
numeric_cols_to_impute = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Domain-specific imputation for TotalCharges
mask_total_charges_nan = df_outlier['TotalCharges'].isnull()
if mask_total_charges_nan.any():
    # Fill based on MonthlyCharges * tenure where possible
    df_outlier.loc[mask_total_charges_nan, 'TotalCharges'] = \
        df_outlier.loc[mask_total_charges_nan, 'MonthlyCharges'] * df_outlier.loc[mask_total_charges_nan, 'tenure']

# Statistical imputation for remaining NaNs in numeric columns
for col in numeric_cols_to_impute:
    if df_outlier[col].isnull().any():
        # Use median for skewed distributions, mean for normal
        if not df_outlier[col].empty and pd.api.types.is_numeric_dtype(df_outlier[col]):
            if df_outlier[col].skew() > 1 or df_outlier[col].skew() < -1:
                df_outlier[col].fillna(df_outlier[col].median(), inplace=True)
            else:
                df_outlier[col].fillna(df_outlier[col].mean(), inplace=True)
        else:
            # Fallback for unexpected empty or non-numeric after previous steps
            df_outlier[col].fillna(0, inplace=True) # or a more suitable default


# Check if any non-numeric types slipped through for critical columns
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    if not pd.api.types.is_numeric_dtype(df_outlier[col]):
        raise TypeError(f"Column '{col}' is not numeric after preprocessing. Current dtype: {df_outlier[col].dtype}")

# --- FIX END ---

outlier_summary = {}

# Method 2A: IQR (Interquartile Range) - Most Common
print('\n2A. IQR Method (1.5 × IQR):')
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    Q1, Q3 = df_outlier[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df_outlier[col] < lower) | (df_outlier[col] > upper)).sum()
    outlier_summary[f'{col}_IQR'] = outliers
    # Only clip if IQR is not NaN (i.e., column has variation)
    if not np.isnan(IQR):
        df_outlier[col] = df_outlier[col].clip(lower, upper)
    print(f'  {col}: {outliers} outliers capped')


# Method 2B: Z-Score (Statistical)
print('\n2B. Z-Score Method (|Z| > 3):')
z_outliers = 0
for col in ['MonthlyCharges', 'tenure']:
    # Ensure column is numeric and not empty before zscore
    if not df_outlier[col].empty and pd.api.types.is_numeric_dtype(df_outlier[col]):
        z_scores = np.abs(zscore(df_outlier[col].dropna()))
        threshold = 3
        outliers_z = (z_scores > threshold).sum()
        z_outliers += outliers_z
        outlier_summary[f'{col}_Zscore'] = outliers_z
        print(f'  {col}: {outliers_z} outliers (Z-score > 3)')
    else:
        print(f"  Skipping Z-score for '{col}': column is empty or not numeric (dtype: {df_outlier[col].dtype})")


# Method 2C: Isolation Forest (ML-based)
print('\n2C. Isolation Forest (Unsupervised ML):')
numeric_subset_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
numeric_subset = df_outlier[numeric_subset_cols].copy()

# Final check for dtypes in numeric_subset for IsolationForest
for col in numeric_subset_cols:
    if not pd.api.types.is_numeric_dtype(numeric_subset[col]):
        print(f"  Warning: Column '{col}' in numeric_subset is still not numeric. Attempting conversion.")
        numeric_subset[col] = pd.to_numeric(numeric_subset[col], errors='coerce')
        numeric_subset[col].fillna(numeric_subset[col].median() if not numeric_subset[col].empty else 0, inplace=True)


if not numeric_subset.empty and all(pd.api.types.is_numeric_dtype(numeric_subset[col]) for col in numeric_subset.columns):
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    # Ensure no NaNs remain before fitting IsolationForest
    numeric_subset_filled = numeric_subset.fillna(numeric_subset.mean())
    outlier_preds = iso_forest.fit_predict(numeric_subset_filled)
    iso_outliers = (outlier_preds == -1).sum()
    outlier_summary['Isolation_Forest'] = iso_outliers
    print(f'  Detected {iso_outliers} anomalous records')
else:
    print("  Skipping Isolation Forest: Numeric subset is empty or contains non-numeric data.")


print('\n✓ Outlier detection complete')
print(f'  Summary: {outlier_summary}')


TECHNIQUE 2: ROBUST OUTLIER DETECTION (3 METHODS)

2A. IQR Method (1.5 × IQR):
  tenure: 0 outliers capped
  MonthlyCharges: 0 outliers capped
  TotalCharges: 0 outliers capped

2B. Z-Score Method (|Z| > 3):
  MonthlyCharges: 0 outliers (Z-score > 3)
  tenure: 0 outliers (Z-score > 3)

2C. Isolation Forest (Unsupervised ML):
  Detected 353 anomalous records

✓ Outlier detection complete
  Summary: {'tenure_IQR': np.int64(0), 'MonthlyCharges_IQR': np.int64(0), 'TotalCharges_IQR': np.int64(0), 'MonthlyCharges_Zscore': np.int64(0), 'tenure_Zscore': np.int64(0), 'Isolation_Forest': np.int64(353)}


In [46]:
# ============================================================
# TECHNIQUE 3: ADVANCED DATA SCALING (4 METHODS)
# ============================================================
print('\n' + '='*60)
print('TECHNIQUE 3: ADVANCED FEATURE SCALING (4 METHODS)')
print('='*60)

from sklearn.preprocessing import RobustScaler # Added import for RobustScaler

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_numeric = df_outlier[numeric_features].copy()

# Method 3A: StandardScaler (Mean=0, Std=1)
print('\n3A. StandardScaler (Z-score normalization):')
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X_numeric)
print(f'  Range: [{X_std.min():.3f}, {X_std.max():.3f}]')
print(f'  Mean: {X_std.mean():.6f}, Std: {X_std.std():.6f}')

# Method 3B: RobustScaler (resistant to outliers)
print('\n3B. RobustScaler (Outlier-resistant):')
scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(X_numeric)
print(f'  Range: [{X_robust.min():.3f}, {X_robust.max():.3f}]')
print(f'  Uses IQR instead of Std, better for outliers')

# Method 3C: MinMaxScaler (0 to 1 range)
print('\n3C. MinMaxScaler (0-1 range):')
scaler_minmax = MinMaxScaler()
X_minmax = scaler_minmax.fit_transform(X_numeric)
print(f'  Range: [{X_minmax.min():.3f}, {X_minmax.max():.3f}]')
print(f'  Bounded scaling for algorithms sensitive to scale')

# Method 3D: Manual Log Transform (for right-skewed data)
print('\n3D. Log Transformation (handling skewness):')
X_log = X_numeric.copy()
for col in X_log.columns:
    if (X_log[col] > 0).all():
        skew_before = X_log[col].skew()
        X_log[col] = np.log1p(X_log[col])
        skew_after = X_log[col].skew()
        print(f'  {col}: Skewness {skew_before:.3f} \u2192 {skew_after:.3f}')

print('\n\u2713 Feature scaling complete with 4 methods')



TECHNIQUE 3: ADVANCED FEATURE SCALING (4 METHODS)

3A. StandardScaler (Z-score normalization):
  Range: [-1.546, 2.826]
  Mean: -0.000000, Std: 1.000000

3B. RobustScaler (Outlier-resistant):
  Range: [-0.959, 2.152]
  Uses IQR instead of Std, better for outliers

3C. MinMaxScaler (0-1 range):
  Range: [0.000, 1.000]
  Bounded scaling for algorithms sensitive to scale

3D. Log Transformation (handling skewness):
  MonthlyCharges: Skewness -0.221 → -0.719

✓ Feature scaling complete with 4 methods


In [47]:
# ============================================================
# TECHNIQUE 4: DISTRIBUTION NORMALIZATION (POWER TRANSFORM)
# ============================================================
print('\n' + '='*60)
print('TECHNIQUE 4: DISTRIBUTION NORMALIZATION')
print('='*60)

from sklearn.preprocessing import PowerTransformer, QuantileTransformer

# Method 4A: Yeo-Johnson Power Transform (handles negative values)
print('\n4A. Yeo-Johnson Power Transform:')
pt_yeo = PowerTransformer(method='yeo-johnson', standardize=True)
X_yeo = pt_yeo.fit_transform(X_numeric)
print(f'  Transforms to approximate normal distribution')
print(f'  Original - Tenure Skewness: {X_numeric["tenure"].skew():.3f}')
print(f'  After Transform - Skewness: {pd.DataFrame(X_yeo, columns=numeric_features)["tenure"].skew():.3f}')

# Method 4B: Box-Cox-style transformation
print('\n4B. Box-Cox Transform (for positive data):')
X_boxcox = X_numeric.copy()
for col in ['MonthlyCharges']:
    if (X_numeric[col] > 0).all():
        skew_before = X_boxcox[col].skew()
        X_boxcox[col], lambda_param = stats.boxcox(X_numeric[col])
        print(f'  {col}: Lambda = {lambda_param:.3f}, Skewness: {skew_before:.3f} \u2192 {X_boxcox[col].skew():.3f}')

# Method 4C: Quantile Transformer (non-parametric)
print('\n4C. Quantile Transformer (Non-parametric):')
qt = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt = qt.fit_transform(X_numeric)
print(f'  Maps to uniform then to normal distribution')
print(f'  Robust to extreme outliers')

print('\n\u2713 Distribution normalization complete')



TECHNIQUE 4: DISTRIBUTION NORMALIZATION

4A. Yeo-Johnson Power Transform:
  Transforms to approximate normal distribution
  Original - Tenure Skewness: 0.240
  After Transform - Skewness: -0.243

4B. Box-Cox Transform (for positive data):
  MonthlyCharges: Lambda = 0.919, Skewness: -0.221 → -0.264

4C. Quantile Transformer (Non-parametric):
  Maps to uniform then to normal distribution
  Robust to extreme outliers

✓ Distribution normalization complete


In [48]:
# ============================================================
# TECHNIQUE 5: DIMENSIONALITY & MULTICOLLINEARITY ANALYSIS
# ============================================================
print('\n' + '='*60)
print('TECHNIQUE 5: MULTICOLLINEARITY & DIMENSIONALITY')
print('='*60)

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.decomposition import PCA # Added import for PCA

# Method 5A: Variance Inflation Factor (VIF)
print('\n5A. Variance Inflation Factor (VIF) Analysis:')
X_scaled = StandardScaler().fit_transform(X_numeric)
vif_data = pd.DataFrame()
vif_data['Feature'] = numeric_features
vif_data['VIF'] = [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
print('\nVIF Values (>10 indicates high multicollinearity):')
print(vif_data.to_string(index=False))

# Method 5B: Correlation-based Feature Removal
print('\n5B. Correlation Analysis & Feature Pruning:')
corr_matrix = X_numeric.corr()
print('\nCorrelation Matrix:')
print(corr_matrix.round(3))
high_corr = np.where(np.abs(corr_matrix) > 0.7)
high_corr = [(corr_matrix.index[x], corr_matrix.columns[y], corr_matrix.iloc[x, y])
             for x, y in zip(*high_corr) if x < y]
if high_corr:
    print(f'\nHighly correlated pairs (>0.7):')
    for feat1, feat2, corr_val in high_corr:
        print(f'  {feat1} ↔ {feat2}: {corr_val:.3f}')

# Method 5C: PCA for Dimensionality Reduction
print('\n5C. PCA (Dimensionality Reduction):')
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
explained_var = np.cumsum(pca.explained_variance_ratio_)
print(f'  Explained Variance by Component:')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'    PC{i+1}: {var:.3f} (Cumulative: {explained_var[i]:.3f})')
print(f'  Components to explain 95% variance: {np.argmax(explained_var >= 0.95) + 1}')

print('\n✓ Multicollinearity analysis complete')



TECHNIQUE 5: MULTICOLLINEARITY & DIMENSIONALITY

5A. Variance Inflation Factor (VIF) Analysis:

VIF Values (>10 indicates high multicollinearity):
       Feature      VIF
        tenure 5.836728
MonthlyCharges 3.216730
  TotalCharges 9.510931

5B. Correlation Analysis & Feature Pruning:

Correlation Matrix:
                tenure  MonthlyCharges  TotalCharges
tenure           1.000           0.248         0.826
MonthlyCharges   0.248           1.000         0.651
TotalCharges     0.826           0.651         1.000

Highly correlated pairs (>0.7):
  tenure ↔ TotalCharges: 0.826

5C. PCA (Dimensionality Reduction):
  Explained Variance by Component:
    PC1: 0.727 (Cumulative: 0.727)
    PC2: 0.253 (Cumulative: 0.980)
    PC3: 0.020 (Cumulative: 1.000)
  Components to explain 95% variance: 2

✓ Multicollinearity analysis complete


In [49]:
# ============================================================
# TECHNIQUE 6: SKEWNESS & KURTOSIS CORRECTION
# ============================================================
print('\n' + '='*60)
print('TECHNIQUE 6: SKEWNESS & KURTOSIS ANALYSIS & CORRECTION')
print('='*60)

print('\nBefore Transformation:')
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    skewness = X_numeric[col].skew()
    kurtosis = X_numeric[col].kurtosis()
    print(f'{col}:')
    print(f'  Skewness: {skewness:.3f} (High if |SK| > 1)', end='')
    if abs(skewness) > 1:
        print(' ← HIGH')
    else:
        print()
    print(f'  Kurtosis: {kurtosis:.3f}\n')

# Apply transformations
df_transformed = X_numeric.copy()

# Square root for moderate skew
df_transformed['tenure_sqrt'] = np.sqrt(df_transformed['tenure'])
# Log for high skew
df_transformed['MonthlyCharges_log'] = np.log1p(df_transformed['MonthlyCharges'])
# Reciprocal for extreme skew
df_transformed['TotalCharges_reciprocal'] = 1 / (df_transformed['TotalCharges'] + 1)

print('\nAfter Transformation:')
print(f'tenure_sqrt skewness: {df_transformed["tenure_sqrt"].skew():.3f}')
print(f'MonthlyCharges_log skewness: {df_transformed["MonthlyCharges_log"].skew():.3f}')
print(f'TotalCharges_reciprocal skewness: {df_transformed["TotalCharges_reciprocal"].skew():.3f}')

print('\n✓ Skewness correction complete')



TECHNIQUE 6: SKEWNESS & KURTOSIS ANALYSIS & CORRECTION

Before Transformation:
tenure:
  Skewness: 0.240 (High if |SK| > 1)
  Kurtosis: -1.387

MonthlyCharges:
  Skewness: -0.221 (High if |SK| > 1)
  Kurtosis: -1.257

TotalCharges:
  Skewness: 0.963 (High if |SK| > 1)
  Kurtosis: -0.229


After Transformation:
tenure_sqrt skewness: -0.236
MonthlyCharges_log skewness: -0.719
TotalCharges_reciprocal skewness: 23.555

✓ Skewness correction complete


In [50]:
print('\n' + '='*60)
print('PREPROCESSING SUMMARY')
print('='*60)
print('''\n6 ADVANCED PREPROCESSING TECHNIQUES APPLIED:
1.  Multiple Imputation (Statistical, Domain-specific, Forward Fill)
2.  Robust Outlier Detection (IQR, Z-score, Isolation Forest)
3.  Advanced Scaling (Standard, Robust, MinMax, Log Transform)
4.  Distribution Normalization (Yeo-Johnson, Box-Cox, Quantile)
5.  Multicollinearity Analysis (VIF, Correlation, PCA)
6.  Skewness & Kurtosis Correction (Sqrt, Log, Reciprocal)

''')




PREPROCESSING SUMMARY

6 ADVANCED PREPROCESSING TECHNIQUES APPLIED:
1.  Multiple Imputation (Statistical, Domain-specific, Forward Fill)
2.  Robust Outlier Detection (IQR, Z-score, Isolation Forest)
3.  Advanced Scaling (Standard, Robust, MinMax, Log Transform)
4.  Distribution Normalization (Yeo-Johnson, Box-Cox, Quantile)
5.  Multicollinearity Analysis (VIF, Correlation, PCA)
6.  Skewness & Kurtosis Correction (Sqrt, Log, Reciprocal)




> **Interpretation.** The dataset is clean: only 11 `TotalCharges` values were missing (new customers with zero tenure), and the IQR and Z-score rules found **no** outliers in the numeric features. Isolation Forest flagged 353 unusual combinations (5 %), which are plausible customers rather than errors, so they were kept. `tenure` and `TotalCharges` are strongly correlated (r = 0.83, VIF 9.5), and two principal components explain 98 % of the numeric variance. Not every transformation helps: the reciprocal transform made `TotalCharges` far more skewed (0.96 → 23.6), so it is not used for modelling.

---

[← Data Loading & SQL Pipeline](01_data_and_sql_pipeline.ipynb) · [Next: Feature Engineering & Exploratory Analysis →](03_feature_engineering_and_eda.ipynb)